
# Define the name of your input and output files
input_filename = r'Datasets\India Agriculture Crop Production.csv\India Agriculture Crop Production.csv'
output_filename = 'odisha_crop_production.csv'

try:
    # Step 1: Import the full dataset
    print(f"Reading the large dataset: '{input_filename}'...")
    df = pd.read_csv(input_filename)
    print("Dataset loaded successfully.")

    # Step 2: Filter the data for the state of Odisha
    print("Filtering for the state of 'Odisha'...")
    # The .copy() is used to avoid a common warning in pandas
    odisha_df = df[df['State'] == 'Odisha'].copy()

    # Step 3: Save the filtered data to a new CSV file
    print(f"Saving the filtered data to '{output_filename}'...")
    # index=False prevents pandas from writing the DataFrame index as a column
    odisha_df.to_csv(output_filename, index=False)

    print("\n✅ Success!")
    print(f"A new file named '{output_filename}' has been created with {len(odisha_df)} rows of data for Odisha.")

except FileNotFoundError:
    print(f"Error: The file '{input_filename}' was not found. Please make sure it's in the same directory as the script.")
except Exception as e:
    print(f"An error occurred: {e}")

# Define the filename for your filtered dataset
filename = 'odisha_crop_production.csv'

# Step 1: Load your Odisha-only dataset
print(f"Reading your filtered dataset: '{filename}'...")
odisha_df = pd.read_csv(filename)
print("Dataset loaded successfully.")

# Step 2: Perform Final Cleaning
print("Cleaning the data...")
# Drop rows where 'Production' is missing (if any)
odisha_df.dropna(subset=['Production'], inplace=True)
# Ensure Area and Production are numeric and positive
odisha_df['Area'] = pd.to_numeric(odisha_df['Area'], errors='coerce')
odisha_df['Production'] = pd.to_numeric(odisha_df['Production'], errors='coerce')
odisha_df.dropna(subset=['Area', 'Production'], inplace=True)
odisha_df = odisha_df[(odisha_df['Area'] > 0) & (odisha_df['Production'] > 0)]

# Recalculate Yield
odisha_df['Yield'] = odisha_df['Production'] / odisha_df['Area']

print(f"Cleaned dataset contains {len(odisha_df)} rows.")

# Step 3: Identify Unique Districts and Years for Weather Data Fetching
unique_districts = odisha_df['District'].unique()
# Handle the 'Year' column format (e.g., '2001-02') to get the starting year
odisha_df['Year_Start'] = odisha_df['Year'].apply(lambda x: int(str(x).split('-')[0]))
min_year = odisha_df['Year_Start'].min()
max_year = odisha_df['Year_Start'].max()

print("\n✅ Analysis Complete!")
print("--------------------------------------------------")
print(f"Data covers the period from {min_year} to {max_year}.")
print(f"Found {len(unique_districts)} unique districts in Odisha.")
print("\nList of Districts:")
# Print districts in a clean, multi-column format for readability
print('\n'.join(['\t'.join(map(str, unique_districts[i:i+3])) for i in range(0, len(unique_districts), 3)]))
print("--------------------------------------------------")
print("\nNext, we will use this information to fetch historical weather data for each of these districts.")

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import joblib # Library for saving and loading models

# --- Step 1: Load and Prepare Data ---
filename = 'odisha_weather_crop_data.csv'
print(f"Reading dataset: '{filename}'...")
final_df = pd.read_csv(filename)

# Clean column names and data
final_df.columns = final_df.columns.str.strip()
numeric_cols = ['Area', 'Production', 'Yield', 'Avg_Temp', 'Total_Precipitation']
for col in numeric_cols:
    final_df[col] = pd.to_numeric(final_df[col], errors='coerce')
final_df.dropna(subset=numeric_cols, inplace=True)
print("Data loading and cleaning complete.")

# --- Step 2: Feature Engineering ---
final_df_encoded = pd.get_dummies(final_df, columns=['District', 'Crop', 'Season'], drop_first=True)
X = final_df_encoded.drop(columns=['State', 'Year', 'Production', 'Yield', 'Area Units', 'Production Units'])
y = final_df_encoded['Yield']

# --- Step 3: Train the Model ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
rf_model_final = RandomForestRegressor(n_estimators=100, random_state=42, oob_score=True, n_jobs=-1)
rf_model_final.fit(X_train, y_train)
print("Model training complete.")

# --- Step 4: Save the Model and Columns ---
model_filename = 'crop_yield_model.joblib'
columns_filename = 'model_columns.joblib'

joblib.dump(rf_model_final, 'crop_yield_model.joblib', compress=3)
joblib.dump(X.columns, columns_filename)

print(f"\n✅ Model saved as '{model_filename}'")
print(f"✅ Model columns saved as '{columns_filename}'")

Reading dataset: 'odisha_weather_crop_data.csv'...
Data loading and cleaning complete.
Model training complete.

✅ Model saved as 'crop_yield_model.joblib'
✅ Model columns saved as 'model_columns.joblib'


In [2]:
import pandas as pd
import joblib

def predict_yield(district, crop, season, area, avg_temp, total_precip):
    """
    Predicts crop yield based on user inputs and the trained model.
    """
    # Load the saved model and the column list
    model = joblib.load('crop_yield_model.joblib')
    model_columns = joblib.load('model_columns.joblib')

    # --- Create an input DataFrame ---
    # Start with a DataFrame of zeros with the correct columns and order
    input_df = pd.DataFrame(columns=model_columns)
    input_df.loc[0] = 0

    # Set the numerical feature values
    input_df['Area'] = area
    input_df['Avg_Temp'] = avg_temp
    input_df['Total_Precipitation'] = total_precip

    # --- Set the categorical feature values (one-hot encoding) ---
    # Construct the column names for the categorical features
    district_col = 'District_' + district
    crop_col = 'Crop_' + crop
    season_col = 'Season_' + season

    # Set the appropriate column to 1 if it exists in the model's columns
    if district_col in model_columns:
        input_df[district_col] = 1
    if crop_col in model_columns:
        input_df[crop_col] = 1
    if season_col in model_columns:
        input_df[season_col] = 1
        
    # --- Make the prediction ---
    prediction = model.predict(input_df)[0]
    
    return prediction

# --- EXAMPLE USAGE ---
if __name__ == '__main__':
    # Let's predict the yield for Rice in Cuttack
    
    # Inputs for the prediction
    example_district = 'CUTTACK'
    example_crop = 'Rice'
    example_season = 'Kharif'
    example_area_hectares = 10.0 # e.g., a 10-hectare farm
    
    # For weather, we'll use historical averages as a stand-in for a forecast
    # You would replace these with actual forecasted values in a real app
    example_avg_temp = 28.5 # Typical average temperature in Celsius
    example_total_precip = 1200 # Typical total rainfall in mm
    
    # Get the prediction
    predicted_yield = predict_yield(
        district=example_district,
        crop=example_crop,
        season=example_season,
        area=example_area_hectares,
        avg_temp=example_avg_temp,
        total_precip=example_total_precip
    )
    
    print("\n--- Yield Prediction ---")
    print(f"District: {example_district}")
    print(f"Crop: {example_crop}")
    print(f"Season: {example_season}")
    print(f"Area: {example_area_hectares} Hectares")
    print("---------------------------------")
    print(f"Predicted Yield: {predicted_yield:.2f} Tonnes per Hectare")


--- Yield Prediction ---
District: CUTTACK
Crop: Rice
Season: Kharif
Area: 10.0 Hectares
---------------------------------
Predicted Yield: 1.59 Tonnes per Hectare


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# --- Step 1: Load Your Final, Fully-Enriched Dataset ---
filename = 'odisha_final_data.csv'
print(f"Reading your final dataset: '{filename}'...")
final_df = pd.read_csv(filename)
print("Dataset loaded successfully.")

# --- Step 2: Final Data Cleaning ---
# Drop any rows that might have missing values from the merge
final_df.dropna(inplace=True)
# Ensure key columns are numeric
numeric_cols = ['Area', 'Production', 'Yield', 'Avg_Temp', 'Total_Precipitation', 'Nitrogen', 'Potassium', 'Phosphorous']
for col in numeric_cols:
    final_df[col] = pd.to_numeric(final_df[col], errors='coerce')
final_df.dropna(subset=numeric_cols, inplace=True)

print(f"Cleaned dataset has {len(final_df)} rows.")

# --- Step 3: Feature Engineering and Selection ---
final_df_encoded = pd.get_dummies(final_df, columns=['District', 'Crop', 'Season'], drop_first=True)

# Define our features (X) and target (y)
# We now include N, P, K in our model
X = final_df_encoded.drop(columns=['State', 'Year', 'Production', 'Yield', 'Area Units', 'Production Units', 'Crop_Category'])
y = final_df_encoded['Yield']

# --- Step 4: Split the Data ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"\nTraining the model on {len(X_train)} samples and testing on {len(X_test)} samples.")

# --- Step 5: Train the Final Random Forest Model ---
rf_model_final = RandomForestRegressor(n_estimators=100, random_state=42, oob_score=True, n_jobs=-1)
rf_model_final.fit(X_train, y_train)
print("Model training complete.")

# --- Step 6: Evaluate the Model's Performance ---
y_pred = rf_model_final.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("\n--- Final Model Evaluation Results ---")
print(f"R-squared (R²): {r2:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"Out-of-Bag (OOB) Score: {rf_model_final.oob_score_:.4f}")

# --- Step 7: Identify the Most Important Factors ---
feature_importances = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model_final.feature_importances_
}).sort_values('importance', ascending=False)

print("\n--- Top 15 Most Important Factors for Predicting Crop Yield ---")
print(feature_importances.head(15))

Reading your final dataset: 'odisha_final_data.csv'...
Dataset loaded successfully.
Cleaned dataset has 13811 rows.

Training the model on 11048 samples and testing on 2763 samples.
Model training complete.

--- Final Model Evaluation Results ---
R-squared (R²): 0.9513
Root Mean Squared Error (RMSE): 2.7001
Out-of-Bag (OOB) Score: 0.9485

--- Top 15 Most Important Factors for Predicting Crop Yield ---
                    feature  importance
59           Crop_Sugarcane    0.928193
0                      Area    0.017782
1                  Avg_Temp    0.013881
2       Total_Precipitation    0.011304
24         District_KORAPUT    0.002901
34      District_SUNDARGARH    0.002708
8          District_BARGARH    0.002358
27     District_NABARANGPUR    0.002147
15          District_GANJAM    0.001757
17         District_JAJAPUR    0.001652
18      District_JHARSUGUDA    0.001520
19       District_KALAHANDI    0.001359
7        District_BALESHWAR    0.001146
16  District_JAGATSINGHAPUR    0.00

In [7]:
import pandas as pd
import joblib
import requests
from geopy.geocoders import Nominatim

# --- PASTE YOUR API KEY HERE ---
OPENWEATHERMAP_API_KEY = "6e0f1433379b9e46dbdb24b115687f15"


def get_weather_forecast(lat, lon):
    """Fetches real-time weather data from OpenWeatherMap."""
    if OPENWEATHERMAP_API_KEY == "YOUR_FREE_API_KEY_GOES_HERE":
        return None, None # Return None if key is not set

    base_url = f"https://api.openweathermap.org/data/2.5/weather"
    params = {
        "lat": lat,
        "lon": lon,
        "appid": OPENWEATHERMAP_API_KEY,
        "units": "metric" # To get temperature in Celsius
    }
    try:
        response = requests.get(base_url, params=params)
        response.raise_for_status()
        data = response.json()
        
        # Extract temperature and assume a default precipitation for this example
        # A real app might use a more detailed forecast API
        avg_temp = data['main']['temp']
        # The free API gives current rain, not a seasonal forecast. 
        # We will use a default value, but show the live temp.
        total_precip = 1200 # Using a typical Kharif season default
        
        return avg_temp, total_precip
    except requests.exceptions.RequestException as e:
        print(f"Weather API request failed: {e}")
        return None, None


def predict_yield(district, crop, season, area, avg_temp, total_precip, nitrogen, potassium, phosphorous):
    """Predicts crop yield using the trained model (from previous steps)."""
    model = joblib.load('crop_yield_model.joblib')
    model_columns = joblib.load('model_columns.joblib')
    
    input_df = pd.DataFrame(columns=model_columns)
    input_df.loc[0] = 0

    input_df['Area'] = area
    input_df['Avg_Temp'] = avg_temp
    input_df['Total_Precipitation'] = total_precip
    input_df['Nitrogen'] = nitrogen
    input_df['Potassium'] = potassium
    input_df['Phosphorous'] = phosphorous

    district_col, crop_col, season_col = f'District_{district}', f'Crop_{crop}', f'Season_{season}'
    if district_col in model_columns: input_df[district_col] = 1
    if crop_col in model_columns: input_df[crop_col] = 1
    if season_col in model_columns: input_df[season_col] = 1
        
    prediction = model.predict(input_df[model_columns])[0]
    return prediction


# --- Main Function to Run the Prediction ---
if __name__ == '__main__':
    
    # --- Farmer's Inputs ---
    farmer_location = "Bhubaneswar, Odisha" # The farmer can enter any city/town
    example_district = 'KHORDHA' # The district this location is in
    example_crop = 'Rice'
    example_season = 'Kharif'
    example_area_hectares = 15.0
    
    # Soil nutrient data for this crop category
    example_nitrogen = 18.0
    example_potassium = 3.9
    example_phosphorous = 18.9

    # --- Step A: Get Coordinates from Location Name ---
    print(f"Finding coordinates for {farmer_location}...")
    geolocator = Nominatim(user_agent="crop_predictor")
    location = geolocator.geocode(farmer_location)
    
    if location:
        print(f"Location found: Lat {location.latitude:.2f}, Lon {location.longitude:.2f}")
        
        # --- Step B: Get Live Weather Data ---
        print("Fetching live weather data...")
        current_temp, forecast_precip = get_weather_forecast(location.latitude, location.longitude)
        
        if current_temp is not None:
            print(f"Current temperature is {current_temp}°C.")
            
            # --- Step C: Make the Prediction ---
            predicted_yield = predict_yield(
                district=example_district,
                crop=example_crop,
                season=example_season,
                area=example_area_hectares,
                avg_temp=current_temp, # Using live temperature!
                total_precip=forecast_precip, # Using default precipitation
                nitrogen=example_nitrogen,
                potassium=example_potassium,
                phosphorous=example_phosphorous
            )
            

            print("\n--- Yield Prediction ---")
            print(f"Predicted Yield for {example_crop} in {example_district}: {predicted_yield:.2f} Tonnes per Hectare")
            
        else:
            print("Could not retrieve weather data. Please check your API key.")
    else:
        print(f"Could not find coordinates for the location: {farmer_location}")

Finding coordinates for Bhubaneswar, Odisha...
Location found: Lat 20.26, Lon 85.84
Fetching live weather data...
Current temperature is 27°C.

--- Yield Prediction ---
Predicted Yield for Rice in KHORDHA: 1.31 Tonnes per Hectare
